In [1]:
# !pip install ruamel.yaml
# cd /shared_data0/weiqiuy/github/raso
# pip install -e .

In [21]:
import torch
from PIL import Image
from raso.models import raso
from raso import inference_ram, get_transform

# Load model
model = raso(pretrained='/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot.pth',
             image_size=384,
             vit='swin_l')
model.eval()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)
transform = get_transform(image_size=384)

# Load and preprocess image
image_path = "/shared_data0/weiqiuy/real_drs/data/abdomen_exlib/images/cholec80_video20_006.png"
image_pil = Image.open(image_path)
image = transform(image_pil).unsqueeze(0).to(device)

result = inference_ram(image, model)
print("Results with default threshold (0.65):", result[0])

/opt/conda/envs/rapids/lib/python3.10/site-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/opt/conda/envs/rapids/lib/python3.10/site-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore
BertLMHeadModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning 

/encoder/layer/0/crossattention/self/query is tied
/encoder/layer/0/crossattention/self/key is tied
/encoder/layer/0/crossattention/self/value is tied
/encoder/layer/0/crossattention/output/dense is tied
/encoder/layer/0/crossattention/output/LayerNorm is tied
/encoder/layer/0/intermediate/dense is tied
/encoder/layer/0/output/dense is tied
/encoder/layer/0/output/LayerNorm is tied
/encoder/layer/1/crossattention/self/query is tied
/encoder/layer/1/crossattention/self/key is tied
/encoder/layer/1/crossattention/self/value is tied
/encoder/layer/1/crossattention/output/dense is tied
/encoder/layer/1/crossattention/output/LayerNorm is tied
/encoder/layer/1/intermediate/dense is tied
/encoder/layer/1/output/dense is tied
/encoder/layer/1/output/LayerNorm is tied
--------------
/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot.pth
--------------


RuntimeError: Error(s) in loading state_dict for RASO:
	size mismatch for label_embed: copying a param with shape torch.Size([2066, 512]) from checkpoint, the shape in current model is torch.Size([2086, 512]).

# Generate embeddings

In [1]:
from transformers import CLIPModel, CLIPProcessor

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
import torch

def save_new_model_weights(
    old_weights_filepath, 
    new_weights_filepath, 
    new_labels_path,
):
    old_weights = torch.load(old_weights_filepath, weights_only=False)
    
    with open(new_labels_path, 'rt') as input_file:
        new_labels = [line.strip() for line in input_file.readlines()]

    # Tokenize text
    inputs = processor(
        text=new_labels,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # Forward pass through CLIP
    with torch.no_grad():
        outputs = model.get_text_features(**inputs)   # <--- use this instead of encode_text

    new_label_embeds = outputs  # shape: (num_labels, hidden_dim)
    
    old_weights['model']['label_embed'] = new_label_embeds
    torch.save(old_weights, new_weights_filepath)

In [5]:
save_new_model_weights(
    old_weights_filepath='/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot.pth',
    new_weights_filepath='/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot_cholecseg8k.pth',
    new_labels_path='/shared_data0/weiqiuy/github/raso/raso/labels_cholecseg8k.txt',
)

In [6]:
save_new_model_weights(
    old_weights_filepath='/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot.pth',
    new_weights_filepath='/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot_cholec_organs.pth',
    new_labels_path='/shared_data0/weiqiuy/github/raso/raso/labels_cholec_organs.txt',
)

In [3]:
save_new_model_weights(
    old_weights_filepath='/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot.pth',
    new_weights_filepath='/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot_cholec_gonogo.pth',
    new_labels_path='/shared_data0/weiqiuy/github/raso/raso/labels_cholec_gonogo.txt',
)

# Load dataset specific model

In [1]:
# cholecseg8k

import torch
from PIL import Image
from raso.models import raso
from raso import inference_ram, get_transform

# Load model
model = raso(pretrained='/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot_cholecseg8k.pth',
             image_size=384,
             vit='swin_l',
             tag_list='/shared_data0/weiqiuy/github/raso/raso/labels_cholecseg8k.txt'
            )
model.eval()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)
transform = get_transform(image_size=384)

# Load and preprocess image
image_path = "/shared_data0/weiqiuy/real_drs/data/abdomen_exlib/images/cholec80_video20_006.png"
image_pil = Image.open(image_path)
image = transform(image_pil).unsqueeze(0).to(device)

result = inference_ram(image, model)
print("Results with default threshold (0.65):", result[0])

/opt/conda/envs/rapids/lib/python3.10/site-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/opt/conda/envs/rapids/lib/python3.10/site-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_bwd(orig_func)  # type: ignore
BertLMHeadModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning 

/encoder/layer/0/crossattention/self/query is tied
/encoder/layer/0/crossattention/self/key is tied
/encoder/layer/0/crossattention/self/value is tied
/encoder/layer/0/crossattention/output/dense is tied
/encoder/layer/0/crossattention/output/LayerNorm is tied
/encoder/layer/0/intermediate/dense is tied
/encoder/layer/0/output/dense is tied
/encoder/layer/0/output/LayerNorm is tied
/encoder/layer/1/crossattention/self/query is tied
/encoder/layer/1/crossattention/self/key is tied
/encoder/layer/1/crossattention/self/value is tied
/encoder/layer/1/crossattention/output/dense is tied
/encoder/layer/1/crossattention/output/LayerNorm is tied
/encoder/layer/1/intermediate/dense is tied
/encoder/layer/1/output/dense is tied
/encoder/layer/1/output/LayerNorm is tied
--------------
/shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot_cholecseg8k.pth
--------------
load checkpoint from /shared_data0/weiqiuy/github/hf_repos/raso/raso_zeroshot_cholecseg8k.pth
Results with default threshold (0